> **Gabarito.** Abra só depois de tentar resolver a versão do aluno.

# Aula 10 – SHAP, rastreabilidade e organização

**Objetivo:** explicar o modelo e deixar o projeto reprodutível e auditável.

Se o SHAP não estiver instalado: `pip install shap`.

## Preparação
Rode esta célula primeiro. Ela importa as funções das aulas anteriores.

In [ ]:
import sys
from pathlib import Path
# Encontra a pasta aulas/ (onde está curso_utils.py), rodando de aulas/ ou de aulas/solucoes/
_aqui = Path.cwd()
_aulas = _aqui if (_aqui / 'curso_utils.py').exists() else _aqui.parent
sys.path.insert(0, str(_aulas))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curso_utils import preparar_nir, pca_svd, preparar_pastas, OUT_DIR, FIG_DIR
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
preparar_pastas()

meta, X, wl = preparar_nir(n=1500)
y = pd.to_numeric(meta['Clay_gkg'], errors='coerce').to_numpy(float)
ok = np.isfinite(y)
X, y = X[ok], y[ok]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

## Exercício 1 – Importância das bandas com SHAP

In [ ]:
import shap
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X_te[:200])
imp = np.abs(sv).mean(0)
top = np.argsort(imp)[::-1][:20][::-1]
plt.barh([f'{wl[i]:.0f} nm' for i in top], imp[top])
plt.xlabel('Média de |SHAP|'); plt.title('Bandas mais importantes – argila'); plt.tight_layout()
plt.savefig(FIG_DIR / '10_shap_argila.png', dpi=150); plt.show()

In [ ]:
# Compare com os loadings do PC1
plt.plot(wl, imp / imp.max(), label='SHAP (normalizado)')
plt.plot(wl, np.abs(pca_svd(X, 1)[1][0]) / np.abs(pca_svd(X, 1)[1][0]).max(), label='|loading PC1| (normalizado)', alpha=0.7)
plt.legend(); plt.xlabel('nm'); plt.show()

## Exercício 2 – Carimbo de execução

In [ ]:
import json, sys
from datetime import datetime

def run_stamp(etapa, payload):
    return {'stage': etapa, 'timestamp': datetime.now().isoformat(), 'python': sys.version, **payload}

def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

write_json(OUT_DIR / '10_modelo_metadata.json', run_stamp('10_shap', {
    'modelo': 'RandomForest', 'n_treino': len(y_tr), 'n_teste': len(y_te),
    'figura': str(FIG_DIR / '10_shap_argila.png'),
}))
print((OUT_DIR / '10_modelo_metadata.json').read_text(encoding='utf-8'))

## Exercício 3 – Refatoração
Abra `curso_utils.py` e compare cada função com a que você escreveu nas aulas 1 a 9. Crie o seu próprio `meu_utils.py` com as suas versões e reescreva a aula 9 importando apenas dele.

## Projeto final
Rode o pipeline completo (aulas 2 a 10) num conjunto de dados novo, alterando apenas a configuração. Entregue os notebooks, o `meu_utils.py`, a pasta `Outputs/` e um relatório de 2 páginas.

## Checkpoint

As bandas mais importantes pelo SHAP coincidem com as absorções conhecidas da argila (cerca de 2200 nm) e da água (1400 e 1900 nm)? O que significaria se o modelo usasse principalmente bandas ruidosas?